<a href="https://colab.research.google.com/github/MateusMaartinS/pythonFag/blob/main/27_Bases_Externas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bases Externas

Nesta aula, aprenderemos como importar dados de fontes externas para nossos projetos. Veremos desde importações simples via URL até técnicas avançadas para contornar bloqueios de servidores e integração com o Google Drive.

---

## 1. Importação via Links (URLs)
A forma mais simples de carregar dados externos no Pandas é passando uma URL diretamente para o método `.read_csv()`.

```python
import pandas as pd

# Link de dados abertos da Receita Federal
link = 'https://www.gov.br/receitafederal/dados/arrecadacao-estado.csv'

# Importante: Definir o separador ';' e o encoding correto para caracteres brasileiros
base_arrecadacao = pd.read_csv(link, sep=';', encoding='iso8859-1')

# Verificando a estrutura dos dados
base_arrecadacao.info()

## 2. Importações Complexas (Web Scraping e Headers)
Alguns servidores bloqueiam acessos automatizados (Erro 403 Forbidden). Para resolver isso, usamos a biblioteca `requests` para simular um navegador real e a `tqdm` para visualizar o progresso de múltiplas requisições.


```python
import pandas as pd
from tqdm.notebook import tqdm
import requests
import io

# Simulando um navegador para evitar erro 403
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

bases = []
estados = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

for estado in tqdm(estados):
    url = f"[http://dadosabertos.ibama.gov.br/dados/SICAFI/](http://dadosabertos.ibama.gov.br/dados/SICAFI/){estado}/Arrecadacao/arrecadacaobenstutelados.csv"
    
    try:
        response = requests.get(url, headers=headers, timeout=20)
        
        if response.status_code == 200:
            # Lendo o texto da resposta como um arquivo
            df = pd.read_csv(io.StringIO(response.text), sep=";", encoding='utf-8')
            df['UF_ORIGEM'] = estado
            bases.append(df)
        else:
            print(f"Erro {response.status_code} ao acessar o estado {estado}")
            
    except Exception as e:
        print(f"Falha ao processar {estado}: {e}")

# Concatenando todas as bases em um único DataFrame
if bases:
    df_final = pd.concat(bases, ignore_index=True)
    print("Download concluído!")
else:
    print("Nenhuma base foi carregada.")
```


## 3. Atividades de Prática
Agora que temos os dados do IBAMA carregados no `df_final`, realize as seguintes operações:

**A) Exploração Inicial**
Visualize as 5 primeiras linhas e a estrutura completa dos dados.
```python
df_final.head()
display(df_final)
```

**B) Manipulação de Strings**
Crie novas colunas de 'dia', 'mes' e 'ano' a partir da coluna 'Data Auto'.
```python
df_final[['dia','mes','ano']] = df_final['Data Auto'].str.split('/', expand=True)
df_final.sample(2)
```

**C) Filtragem Avançada**
Descubra o total de autos únicos para a cidade de **Cascavel/PR** no ano de **2018**.
```python
estado = 'PR'
cidade = 'CASCAVEL'
ano = '2018'

base_estado = df_final[df_final['UF']==estado]
# Nota: Verifique se o nome da coluna no CSV original contém caracteres especiais como 'MunicÃ­pio'
base_cidade = base_estado[base_estado['MunicÃ­pio']==cidade]
base_ano = base_cidade[base_cidade['ano']==ano]

total_autos = base_ano['NÂº AI'].unique().size
print(f'Total de autos únicos em {cidade}: {total_autos}')

## 4. Importação via Google Drive
Caso seus arquivos estejam salvos na nuvem, você pode montar o seu Drive diretamente no ambiente do Colab.

```python
# Solicita permissão para acessar seus arquivos
from google.colab import drive
drive.mount('/drive')

import pandas as pd

# Substitua pelo caminho real do seu arquivo dentro do Drive

caminho = '/drive/My Drive/seu_projeto/dados.xlsx'
dados_drive = pd.read_excel(caminho)

## 5. Configuração de Ambiente Local

Para projetos profissionais fora do Google Colab, utilizamos **Ambientes Virtuais (venv)**. Eles servem para isolar as bibliotecas de cada projeto, evitando que uma atualização no "Projeto A" quebre o "Projeto B".

### Passo 1: Instalação e Verificação Inicial
Antes de criar o ambiente, o Python precisa estar instalado e acessível pelo terminal.

**No Windows:**
1. Baixe o instalador em [python.org](https://www.python.org/).
2. **IMPORTANTE:** No instalador, marque a caixa **"Add Python to PATH"**.
3. Verifique no terminal (PowerShell ou CMD):
   ```powershell
   python --version
   ```

**No Linux (Ubuntu/Debian):**
1. Atualize os repositórios e instale o Python e o suporte a ambientes virtuais:
   ```bash
   sudo apt update
   sudo apt install python3 python3-venv
   ```
2. Verifique no terminal:
   ```bash
   python3 --version
   ```

---

### Passo 2: Criar o Ambiente Virtual
O comando cria uma pasta (geralmente chamada `.venv`) que guardará as bibliotecas do projeto.



**Windows:**
```powershell
python -m venv .venv
```

**Linux:**
```bash
python3 -m venv .venv
```

---

### Passo 3: Ativar o Ambiente
A ativação "avisa" ao seu computador que, a partir de agora, os comandos de Python devem usar a pasta `.venv` que acabamos de criar.

**Windows (PowerShell):**
```powershell
.\.venv\Scripts\Activate.ps1
```

**Linux / WSL / macOS:**
```bash
source .venv/bin/activate
```

*(Você verá o prefixo `(.venv)` aparecer no início da linha do seu terminal).*

---

### Passo 4: Gerenciar Dependências
Com o ambiente **ativado**, você pode gerenciar os pacotes do seu projeto.

**Para instalar bibliotecas:**
```bash
pip install pandas requests tqdm
```

**Para gerar a lista de dependências (Snapshot):**
```bash
pip freeze > requirements.txt
```

**Para instalar as dependências de um projeto existente:**
```bash
pip install -r requirements.txt
```

---

### Passo 5: Sair e Limpeza
Para sair do ambiente e voltar ao Python global do sistema:
```bash
deactivate
```

> **Dica de Ouro:** Nunca envie a pasta `.venv` para o GitHub (ela é pesada e específica para o seu PC). Adicione a linha `.venv/` ao seu arquivo **.gitignore**.

## 6. Fontes de Dados
Para encontrar mais bases de dados para seus projetos, explore:
- [Portal de Dados Abertos do Governo Federal](https://dados.gov.br/home)
- [Base dos Dados](https://basedosdados.org/search)
- [Kaggle Datasets](https://www.kaggle.com/datasets)